# Module 4a: RayDP - Spark on Ray

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers RayDP:
1. Running Spark on Ray infrastructure
2. Zero-copy data transfer between Spark and Ray
3. Unified resource management
4. Complete ETL → ML pipeline

**Prerequisites**: `pip install raydp pyspark`

**Note**: RayDP requires specific version compatibility. Check https://github.com/oap-project/raydp for current versions.

## Key Takeaways

- **RayDP** runs Spark executors as Ray actors
- **Zero-copy** data transfer via shared object store
- **Unified resources**: Single cluster for Spark and Ray
- **ray.data.from_spark()** for seamless handoff

In [ ]:
import ray
import numpy as np
import pandas as pd

# Check if RayDP is available
try:
    import raydp
    RAYDP_AVAILABLE = True
    print(f"RayDP version: {raydp.__version__}")
except ImportError:
    RAYDP_AVAILABLE = False
    print("RayDP not installed. Some examples will use simulation mode.")
    print("Install with: pip install raydp")

---

## 1. Understanding RayDP Architecture

### How RayDP Works

```
┌─────────────────────────────────────────────────────────────────┐
│                      RAY CLUSTER                                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌─────────────────┐                                            │
│  │  SPARK DRIVER   │  (Python process)                         │
│  │  SparkSession   │                                            │
│  └────────┬────────┘                                            │
│           │                                                      │
│           │ (RayDP routes to Ray actors)                        │
│           │                                                      │
│  ┌────────┴────────┬────────────────┬────────────────┐         │
│  ▼                 ▼                ▼                ▼         │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐          │
│  │  Ray     │ │  Ray     │ │  Ray     │ │  Ray     │          │
│  │  Actor   │ │  Actor   │ │  Actor   │ │  Actor   │          │
│  │ (Exec 1) │ │ (Exec 2) │ │ (Exec 3) │ │ (Exec 4) │          │
│  └──────────┘ └──────────┘ └──────────┘ └──────────┘          │
│       │            │            │            │                  │
│       └────────────┴────────────┴────────────┘                  │
│                          │                                       │
│                          ▼                                       │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │               RAY OBJECT STORE (Shared Memory)            │  │
│  │                                                           │  │
│  │   Spark DataFrames ←──zero-copy──→ Ray Datasets          │  │
│  └──────────────────────────────────────────────────────────┘  │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Initialize Ray
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray initialized: {ray.is_initialized()}")
print(f"Available resources: {ray.available_resources()}")

---

## 2. Creating Spark Session with RayDP

In [ ]:
if RAYDP_AVAILABLE:
    # Initialize Spark on Ray
    spark = raydp.init_spark(
        app_name="RayDP_Demo",
        num_executors=2,
        executor_cores=2,
        executor_memory="2GB",
    )
    print(f"Spark version: {spark.version}")
    print(f"Spark running on Ray!")
else:
    # Fallback to regular PySpark for demonstration
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("RayDP_Demo_Fallback") \
        .master("local[2]") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()
    print(f"Spark version: {spark.version}")
    print("(Running in local mode - RayDP not available)")

---

## 3. Spark Operations

In [ ]:
from pyspark.sql import functions as F

# Create sample weather data
np.random.seed(42)
n_records = 10000

weather_pdf = pd.DataFrame({
    "station_id": np.random.choice(["A", "B", "C", "D"], n_records),
    "timestamp": pd.date_range("2023-01-01", periods=n_records, freq="h"),
    "temperature": np.random.uniform(-10, 40, n_records),
    "humidity": np.random.uniform(20, 100, n_records),
    "pressure": np.random.uniform(980, 1040, n_records),
    "wind_speed": np.random.uniform(0, 30, n_records),
    "precipitation": np.random.exponential(2, n_records),
    "quality_flag": np.random.choice(["OK", "SUSPECT", "MISSING"], n_records, p=[0.9, 0.07, 0.03])
})

# Create Spark DataFrame
weather_df = spark.createDataFrame(weather_pdf)
print(f"Created DataFrame with {weather_df.count()} records")
weather_df.printSchema()

In [ ]:
# Spark ETL operations
print("Performing Spark ETL...")

# Filter and clean
cleaned_df = weather_df \
    .filter(F.col("quality_flag") == "OK") \
    .filter(F.col("temperature").between(-50, 60)) \
    .filter(F.col("humidity").between(0, 100))

# Add derived columns
processed_df = cleaned_df \
    .withColumn("date", F.to_date("timestamp")) \
    .withColumn("hour", F.hour("timestamp")) \
    .withColumn("month", F.month("timestamp")) \
    .withColumn("temp_category",
                F.when(F.col("temperature") < 0, "cold")
                .when(F.col("temperature") < 20, "mild")
                .otherwise("warm"))

print(f"After cleaning: {processed_df.count()} records")
processed_df.show(5)

In [ ]:
# Aggregate by station and date
daily_summary = processed_df.groupBy("station_id", "date") \
    .agg(
        F.avg("temperature").alias("avg_temp"),
        F.max("temperature").alias("max_temp"),
        F.min("temperature").alias("min_temp"),
        F.avg("humidity").alias("avg_humidity"),
        F.avg("pressure").alias("avg_pressure"),
        F.sum("precipitation").alias("total_precip"),
        F.count("*").alias("num_readings")
    )

print(f"Daily summaries: {daily_summary.count()} records")
daily_summary.show(5)

---

## 4. Converting to Ray Dataset

### Zero-Copy Transfer (with RayDP)

In [ ]:
if RAYDP_AVAILABLE:
    # Zero-copy transfer with RayDP
    print("Using zero-copy transfer (RayDP)...")
    ray_ds = ray.data.from_spark(daily_summary)
else:
    # Fallback: through pandas
    print("Using pandas intermediary (no RayDP)...")
    pandas_df = daily_summary.toPandas()
    ray_ds = ray.data.from_pandas(pandas_df)

print(f"\nRay Dataset: {ray_ds}")
print(f"Count: {ray_ds.count()}")

In [ ]:
# View Ray Dataset
ray_ds.take(5)

---

## 5. Ray ML Pipeline

In [ ]:
# Feature engineering with Ray Data
def add_ml_features(batch: pd.DataFrame) -> pd.DataFrame:
    """Add features for ML model."""
    result = batch.copy()

    # Temperature range
    result["temp_range"] = result["max_temp"] - result["min_temp"]

    # Humidity-temperature interaction
    result["humidity_temp_ratio"] = result["avg_humidity"] / (result["avg_temp"] + 30)  # Avoid division issues

    # Pressure deviation from standard
    result["pressure_deviation"] = result["avg_pressure"] - 1013.25

    # Reading completeness (24 expected per day)
    result["completeness"] = result["num_readings"] / 24.0

    return result

# Apply feature engineering
ml_ds = ray_ds.map_batches(add_ml_features, batch_format="pandas")

print("With ML features:")
ml_ds.take(3)

In [ ]:
# Prepare for training - drop non-numeric columns
def prepare_for_training(batch: pd.DataFrame) -> pd.DataFrame:
    """Keep only numeric columns for training."""
    # Select feature columns
    feature_cols = [
        "avg_temp", "max_temp", "min_temp", "avg_humidity",
        "avg_pressure", "temp_range", "humidity_temp_ratio",
        "pressure_deviation", "completeness"
    ]
    target_col = "total_precip"

    result = batch[feature_cols + [target_col]].copy()
    return result

training_ds = ml_ds.map_batches(prepare_for_training, batch_format="pandas")

print("Training dataset schema:")
print(training_ds.schema())

In [ ]:
# Train/test split
train_ds, test_ds = training_ds.train_test_split(test_size=0.2)

print(f"Training samples: {train_ds.count()}")
print(f"Test samples: {test_ds.count()}")

In [ ]:
# Train XGBoost model
from ray.train.xgboost import XGBoostTrainer
from ray.train import ScalingConfig

trainer = XGBoostTrainer(
    label_column="total_precip",
    num_boost_round=50,
    params={
        "objective": "reg:squarederror",
        "eval_metric": ["rmse", "mae"],
        "max_depth": 6,
        "eta": 0.1,
        "subsample": 0.8,
    },
    datasets={"train": train_ds, "valid": test_ds},
    scaling_config=ScalingConfig(
        num_workers=2,
        use_gpu=False,
    ),
)

result = trainer.fit()

print("\n" + "="*50)
print("TRAINING RESULTS")
print("="*50)
print(f"RMSE: {result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"MAE: {result.metrics.get('valid-mae', 'N/A'):.4f}")

---

## 6. Performance Benefits of RayDP

In [ ]:
import time

# Compare transfer methods
print("Data Transfer Comparison")
print("="*50)

# Method 1: Through Pandas (baseline)
start = time.time()
pdf = daily_summary.toPandas()
ds_pandas = ray.data.from_pandas(pdf)
_ = ds_pandas.count()  # Force evaluation
pandas_time = time.time() - start
print(f"Through Pandas: {pandas_time:.3f}s")

if RAYDP_AVAILABLE:
    # Method 2: RayDP zero-copy
    start = time.time()
    ds_raydp = ray.data.from_spark(daily_summary)
    _ = ds_raydp.count()  # Force evaluation
    raydp_time = time.time() - start
    print(f"RayDP zero-copy: {raydp_time:.3f}s")
    print(f"\nSpeedup: {pandas_time/raydp_time:.1f}x")
else:
    print("\nRayDP not available for comparison")

---

## 7. Resource Management

RayDP allows fine-grained control over resources:

In [ ]:
# Resource configuration examples
resource_configs = {
    "small": {
        "num_executors": 2,
        "executor_cores": 2,
        "executor_memory": "2GB",
        "use_case": "Development, testing"
    },
    "medium": {
        "num_executors": 4,
        "executor_cores": 4,
        "executor_memory": "8GB",
        "use_case": "Medium datasets (10-100 GB)"
    },
    "large": {
        "num_executors": 8,
        "executor_cores": 8,
        "executor_memory": "16GB",
        "use_case": "Large datasets (100+ GB)"
    },
}

print("RayDP Resource Configurations")
print("="*60)
for name, config in resource_configs.items():
    total_cores = config["num_executors"] * config["executor_cores"]
    total_memory = config["num_executors"] * int(config["executor_memory"].replace("GB", ""))
    print(f"\n{name.upper()}:")
    print(f"  Executors: {config['num_executors']}")
    print(f"  Cores/executor: {config['executor_cores']}")
    print(f"  Memory/executor: {config['executor_memory']}")
    print(f"  Total: {total_cores} cores, {total_memory} GB")
    print(f"  Use case: {config['use_case']}")

---

## Summary

### RayDP Key Points

1. **Unified Infrastructure**: Single Ray cluster for both Spark and ML
2. **Zero-Copy Transfer**: `ray.data.from_spark()` avoids serialization
3. **Resource Sharing**: Spark executors as Ray actors
4. **Best of Both**: Spark SQL + Ray ML in one pipeline

### When to Use RayDP

| Use Case | RayDP? |
|----------|--------|
| New project, need both Spark and Ray | Yes |
| Existing Spark cluster, want Ray ML | Maybe |
| Only need Spark OR Ray | No |
| Memory-constrained environment | Careful |

### Next Steps

- `04b_spark_ray_comparison.ipynb`: Side-by-side framework comparison
- `04c_data_handoff_patterns.ipynb`: Data transfer strategies

In [ ]:
# Cleanup
if RAYDP_AVAILABLE:
    raydp.stop_spark()
else:
    spark.stop()

ray.shutdown()
print("Cleanup complete.")